# Phase 8 · 응용 — PCA로 데이터 요약

데이터의 공분산행렬을 만들고 `np.linalg.eig` 로 주성분(고유벡터)을 구한 뒤, PC1으로 투영해 차원을 줄입니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터** — 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요 — 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch8')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글이 네모로 깨짐)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 우상향으로 길쭉한 데이터 구름
X = np.array([
    [-3, -1.5], [-2.4, -1.4], [-2, -0.7], [-1.4, -1], [-1, -0.2], [-0.6, -0.6],
    [0, 0.1], [0.4, -0.2], [0.8, 0.5], [1.2, 0.4], [1.6, 1.1], [2, 0.8],
    [2.4, 1.5], [3, 1.3], [-1.8, -0.4], [1, 0.9],
], dtype=float)
print('데이터 개수:', len(X))

## 1. 공분산행렬 → 고유분해(주성분)

데이터를 평균 중심으로 옮긴 뒤 공분산행렬을 만들고, 고유값·고유벡터를 구합니다.

In [ ]:
Xc = X - X.mean(axis=0)          # 평균 중심으로
cov = np.cov(Xc, rowvar=False)   # 2x2 공분산행렬
vals, vecs = np.linalg.eig(cov)

# 고유값 큰 순으로 정렬
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print('고유값 =', np.round(vals, 3))
print('PC1 방향 =', np.round(vecs[:, 0], 3))

ratio = vals[0] / vals.sum() * 100
print(f'PC1이 담은 정보 비율 = {ratio:.1f}%')

## 2. 데이터 구름과 주성분 축 그리기

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(Xc[:, 0], Xc[:, 1], color='#475569', label='데이터')
for i, color in zip([0, 1], ['#6366f1', '#ec4899']):
    vec = vecs[:, i] * np.sqrt(vals[i]) * 2.5
    plt.arrow(0, 0, vec[0], vec[1], color=color, width=0.04,
              length_includes_head=True, label=f'PC{i+1}')
plt.axhline(0, color='#e2e8f0'); plt.axvline(0, color='#e2e8f0')
plt.gca().set_aspect('equal'); plt.legend(); plt.title('데이터 구름과 주성분(PC1, PC2)')
plt.show()

## 3. PC1으로 투영 — 2D → 1D 차원축소

각 점을 PC1 방향으로 투영하면 숫자 하나(1D)로 요약됩니다.

In [ ]:
pc1 = vecs[:, 0]
t = Xc @ pc1               # 각 점의 PC1 좌표 (1D 값)
proj = np.outer(t, pc1)    # 다시 2D 위치로 (직선 위)

plt.figure(figsize=(5, 5))
plt.scatter(Xc[:, 0], Xc[:, 1], color='#cbd5e1', label='원본 2D')
plt.scatter(proj[:, 0], proj[:, 1], color='#6366f1', label='PC1 투영 (1D)')
for p, q in zip(Xc, proj):
    plt.plot([p[0], q[0]], [p[1], q[1]], color='#cbd5e1', lw=0.8, ls='--')
plt.gca().set_aspect('equal'); plt.legend(); plt.title('PC1으로 투영 = 차원축소')
plt.show()

print('1D로 요약된 값(앞 5개):', np.round(t[:5], 2))

## 4. 직접 해보기 ✏️

PC1이 담은 정보 비율이 80% 이상인지 확인하세요. (이 데이터는 길쭉해서 PC1 비중이 큼)

In [ ]:
ratio = vals[0] / vals.sum() * 100
print(f'PC1 정보 비율 = {ratio:.1f}%')
assert ratio >= 80, '이 데이터에서는 PC1 비중이 80% 이상이어야 합니다.'
print('✅ 확인! PC1 하나로도 대부분의 정보를 담습니다.')

---
## 🎓 핵심 과정(1~8) 완료

벡터 → 내적 → 행렬 → 연립방정식 → 선형변환 → 행렬식·역행렬 → 고유값 → PCA.
선형대수로 AI의 토대를 갖췄습니다. 더 깊이 가려면 심화(9~12: rank·직교·SVD·그래디언트)로 이어가세요.